## Export S2-only model

In [ ]:
!uv pip install onnx onnxscript requests aiohttp segmentation_models_pytorch fsspec

In [ ]:
import io

import fsspec
import segmentation_models_pytorch as smp
import torch
from torch.export import Dim

EPS = 1e-10


class SKeMaModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        in_channels = 10
        self.model = smp.Unet(
            encoder_name="tu-maxvit_tiny_tf_512",
            in_channels=in_channels,
            encoder_weights=None,
        )

        self.register_buffer(
            "per_channel_mean",
            torch.zeros((1, in_channels, 1, 1)),
        )

        self.register_buffer(
            "per_channel_std",
            torch.ones((1, in_channels, 1, 1)),
        )

    def forward(self, x):
        # Unpack spectral bands
        blue = x.select(1, 0).unsqueeze(1)
        green = x.select(1, 1).unsqueeze(1)
        red = x.select(1, 2).unsqueeze(1)
        nir = x.select(1, 3).unsqueeze(1)
        re = x.select(1, 4).unsqueeze(1)

        # Compute vegetation indices
        ndvi = self.normalized_index(nir, red)
        gndvi = self.normalized_index(nir, green)
        ndvi_re = self.normalized_index(re, red)

        # Compute other indices
        ndwi = self.normalized_index(green, nir)
        chl_green = (nir / (green + EPS)) - 1  # Chlorophyll Index Green

        # Stack all bands and indices
        x_aug = torch.cat([blue, green, red, nir, re, ndvi, ndwi, gndvi, chl_green, ndvi_re], dim=1)

        x_aug_normalized = (x_aug - self.per_channel_mean) / self.per_channel_std

        return self.model(x_aug_normalized)

    @staticmethod
    def normalized_index(a, b):
        return (a - b) / (a + b + EPS)


model = SKeMaModel()

sample_input = torch.rand((2, 5, 512, 512), device=torch.device("cpu"), requires_grad=False)
model(sample_input)

In [ ]:
with fsspec.open("https://huggingface.co/m5ghanba/SKeMa/resolve/main/modelS2Only.pth") as f:
    bytes_io = io.BytesIO(f.read())
state_dict = torch.load(bytes_io, map_location="cpu")

In [ ]:
# Update keys
del state_dict["mean"]
state_dict["per_channel_mean"] = torch.tensor([
    2.08900522e02,
    2.70272557e02,
    1.52312422e02,
    9.87182507e02,
    3.26650321e02,
    5.65647882e-02,
    1.62913280e-01,
    -1.62913280e-01,
    1.90832012e07,
    1.09887867e-01,
]).view(1, -1, 1, 1)

del state_dict["std"]
state_dict["per_channel_std"] = torch.tensor([
    1.59021908e02,
    2.18393833e02,
    2.08355086e02,
    1.29667310e03,
    3.79794112e02,
    6.53314129e-01,
    7.11820223e-01,
    7.11820223e-01,
    2.07060299e10,
    3.90619966e-01,
]).view(1, -1, 1, 1)

model.load_state_dict(state_dict, strict=True)
model.eval()

In [ ]:
outpath = "./Unet_tu-maxvit_tiny_tf_512_20260222.onnx"

img_shape = (Dim("batch"), Dim.STATIC, Dim.STATIC, Dim.STATIC)
args = (sample_input,)

with torch.no_grad():
    torch.onnx.export(
        model,
        args,
        outpath,
        # opset_version=22
        export_params=True,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        # dynamic_shapes=(img_shape,),
        dynamic_axes={
            "input": {0: "batch_size"},
            "output": {0: "batch_size"},
        },
        verbose=True,
        dynamo=False,
    )

## Export S2 + Bathymetry + Substrate model

In [ ]:
EPS = 1e-10


class SKeMaBathyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = smp.Unet(
            encoder_name="tu-maxvit_tiny_tf_512",
            in_channels=12,
            encoder_weights=None,
        )

        self.register_buffer(
            "per_channel_mean",
            torch.tensor([
                2.02127847e02,
                2.64991799e02,
                1.45913497e02,
                9.57456953e02,
                3.20302883e02,
                1.37548690e00,
                -3.29322396e-01,
                3.47405153e01,
                3.66107438e-02,
                1.84492036e-01,
                -1.84492036e-01,
                2.22893410e07,
                9.99078992e-02,
            ]).view(1, -1, 1, 1),
        )

        self.register_buffer(
            "per_channel_std",
            torch.tensor([
                1.61504107e02,
                2.22303637e02,
                2.03997451e02,
                1.26105656e03,
                3.79069759e02,
                1.36767747e00,
                2.02345453e02,
                2.60894243e01,
                6.71879776e-01,
                7.23202999e-01,
                7.23202999e-01,
                2.25708723e10,
                4.06695959e-01,
            ]).view(1, -1, 1, 1),
        )

    def forward(self, x):
        # Unpack spectral bands
        blue = x.select(1, 0).unsqueeze(1)
        green = x.select(1, 1).unsqueeze(1)
        red = x.select(1, 2).unsqueeze(1)
        nir = x.select(1, 3).unsqueeze(1)
        re = x.select(1, 4).unsqueeze(1)
        substrate = x.select(1, 5).unsqueeze(1)
        bathymetry = x.select(1, 6).unsqueeze(1)

        # Compute vegetation indices
        ndvi = self.normalized_index(nir, red)
        gndvi = self.normalized_index(nir, green)
        ndvi_re = self.normalized_index(re, red)

        # Compute other indices
        ndwi = self.normalized_index(green, nir)
        chl_green = (nir / (green + EPS)) - 1  # Chlorophyll Index Green

        # Stack all bands and indices
        x_aug = torch.cat(
            [blue, green, red, nir, re, substrate, bathymetry, ndvi, ndwi, gndvi, chl_green, ndvi_re], dim=1
        )

        x_aug_normalized = (x_aug - self.per_channel_mean) / self.per_channel_std

        return self.model(x_aug_normalized)

    @staticmethod
    def normalized_index(a, b):
        return (a - b) / (a + b + EPS)


model = SKeMaBathyModel()

sample_input = torch.rand((2, 7, 512, 512), device=torch.device("cpu"), requires_grad=False)
model(sample_input)

In [ ]:
ckpt = torch.load("./Unet_tu-maxvit_tiny_tf_512_20250714_222203.ckpt", map_location="cpu")
state_dict = ckpt["state_dict"]

# Update keys
del state_dict["mean"]
del state_dict["std"]
model.load_state_dict(state_dict, strict=False)
model.eval()

In [ ]:
torch.onnx.export(
    model,
    sample_input,
    "./Unet_tu-maxvit_tiny_tf_512_20250714_222203.onnx",
    input_names=["input"],
    output_names=["output"],
    export_params=True,
    external_data=False,  # Store model weights in the model file
    opset_version=15,  # ONNX opset version
    do_constant_folding=True,  # Optimize constants
    verbose=False,
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    # dynamic_shapes={"x": (torch.export.Dim("batch"), 5, 512, 512)},
    dynamo=False,
)